## Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent  processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# Load variables from .env file
load_dotenv()

google_api_key = os.getenv("GOOGLE_API_KEY") or os.getenv("GOOGLE_GENAI_API_KEY")
if google_api_key:
    os.environ["GOOGLE_API_KEY"] = google_api_key

model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")
response = model.invoke("What is the capital of France?")
response.content

### Define structured schema using Pydantic
We define a Pydantic model to represent the structured schema we want the model to output. LangChain's `.with_structured_output` will enforce the model's response to conform to this structure.

In [9]:
from pydantic import BaseModel, Field

# Define the schema using Pydantic
class Book(BaseModel):
    title: str = Field(description="Title of the book")
    author: str = Field(description="Author of the book")
    published_year: int = Field(description="Year the book was published")

# Configure the model to return structured output matching the Pydantic schema
structured_model = model.with_structured_output(Book)

# Invoke the model
response = structured_model.invoke("Tell me about the book 'To Kill a Mockingbird'")
response

Book(title='To Kill a Mockingbird', author='Harper Lee', published_year=1960)

## Message output along with parsed structure.

In [14]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """you have to answer the movie details on movie name"""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year date of movie")
    director: str = Field(..., description="Movie director name")
    rating: float = Field(..., description="Give me true rating of the movie")

model_with_structure = model.with_structured_output(Movie, include_raw = True)

response = model_with_structure.invoke("Give me details of Iron man")
response

{'raw': AIMessage(content=[{'type': 'text', 'text': '{"title":"Iron Man","year":2008,"director":"Jon Favreau","rating":7.9}', 'extras': {'signature': 'Eq8HCqwHAQw51sepPUDhOKf3vX40PWJRgLteucaQdXoQ/HmPzte4qsy2rfXFZWOxXzQcK8V1Y0EDPN7adI6xHncjZXwkouXULxjxQ5A2odWnfx5wIV02YnZUx9SfzxJ7nEGHkRrKcfc8LziAma0l/abkZzNVh/STdo9sj9hBQN1noGj4SeScRuBMYhj6GLHI6uiDBOqN7LZEx1n+CVidTupn62ZBJKhfpV0p3hXkeUv2LslGTR517McuIHnONrwcXMHPRQD3bHeA+qQ7jx96hCMmc2CMziuGQCa3YJHTXxunA8ZUL/gwZL+LiaUtFriDRuOQ6GlLXM5GTRFzW45sN7mOi6CuK4suTs8UylXEYaMFuruiJGdlv7awjd583A22wwSWK8io9cnUpRPWWk1IYqBkZ9n4hM7+xqf81ygbPv7aEwp7ptXgsPcb9i+Dwt/ki3edET4J/rXhRdDCEArCWQ9bep42pA/OKXEMLv9OlvfFi8MLsvitvgoIsu7lxq0NfkHP7FFcy50KiDbjavCxQWnjz5bPKyjA/O3sUGTDi4hXZDWcKon9vfP8pbMdPNA0HUEEOY3lgxpw6UIttOPHslXbEwkDEoq5xWYfMNKGzyP5ixgTPE/o8NJ4WO+MSLJHPCHLRUHdKPU40kT+h5omSYxY2eKyF2RlhgDiN3MJ2zpSl3nEVzgKQQwclNahMxZept2HC3OeKgfz8neOdguS1Y5zGyY6dHo58R0B0fFECkpQkcFyvZXsTSjagGvryczq2QF9LXHzL8YogNtptTxy+hTnvto6zNuSlJpy4VMgaca/whM+KphffnIUcv1POmylATNOA4413U+HwYCZK

## Nested structure

In [15]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Iron man")
response

MovieDetails(title='Iron Man', year=2008, cast=[Actor(name='Robert Downey Jr.', role='Tony Stark / Iron Man'), Actor(name='Terrence Howard', role='James Rhodes'), Actor(name='Jeff Bridges', role='Obadiah Stane'), Actor(name='Gwyneth Paltrow', role='Pepper Potts')], genres=140.0)

# TypedDict
Typedict provide a simpler alternative using python built-in typing ideal when you don't need runtime validation.


In [18]:
from typing_extensions import TypedDict, Annotated

class Moviedict(TypedDict):
    """You have to give movie details"""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's ratting out of 10"]

model_withtypedict =  model.with_structured_output(Moviedict)
response = model_withtypedict.invoke("Tell me about avanger infinity war")
response

{'title': 'Avengers: Infinity War',
 'year': 2018,
 'director': 'Anthony Russo, Joe Russo',
 'rating': 8.4}

In [19]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Iron man")
response

{'title': 'Iron Man',
 'year': 2008,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Terrence Howard', 'role': 'James Rhodes'},
  {'name': 'Jeff Bridges', 'role': 'Obadiah Stane'},
  {'name': 'Gwyneth Paltrow', 'role': 'Pepper Potts'}],
 'genres': 3}

In [22]:
# model structure details
# so this info shows about model ability to do work like gemini-flash-2.0 using and these are the model working from calling model gemini api call from ai studio.
model.profile

{'name': 'Gemini 3 Flash Preview',
 'release_date': '2025-12-17',
 'last_updated': '2025-12-17',
 'open_weights': False,
 'max_input_tokens': 1048576,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': True,
 'pdf_inputs': True,
 'video_inputs': True,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': True,
 'image_url_inputs': True,
 'image_tool_message': True,
 'tool_choice': True}

# DataClasses
In data classes we have class typically containig mainly data, although there aren't really any restictions. You create it using the @dataclasses as decorator.

In [24]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format=ContactInfo # Auto selects provide structure
)

result = agent.invoke({
    "messages": [{"role":"user","content": "Extract contant info from: Ashwin mehta, ashwinmehta1234500@gmail.com, 888444676"}]
})

print(result["structured_response"])

name='Ashwin mehta' email='ashwinmehta1234500@gmail.com' phone='888444676'


In [26]:
# dataclasses

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Call information of the person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format=ContactInfo # Auto selects provide structure
)

result = agent.invoke({
    "messages": [{"role":"user","content": "Extract contant info from: Ashwin mehta, ashwinmehta1234500@gmail.com, 888444676"}]
})

print(result["structured_response"])



ContactInfo(name='Ashwin mehta', email='ashwinmehta1234500@gmail.com', phone='888444676')
